# Pokemon GO Raid Advisor

Practica Diego Rodriguez de AI Engineering.

Agente ReAct hecho con LangChain que ayuda a planear raids de Pokemon GO consultando la [PokeAPI](https://pokeapi.co/). Tiene dos tools (info y movimientos) y decide por su cuenta cuando llamarlas: para datos concretos de un Pokemon usa las tools y razona sobre debilidades y counters; las preguntas generales del juego las contesta directamente.

Stack: `langchain>=1.0`, `langgraph>=1.0`, `langchain-openai`, `gpt-5-mini`, PokeAPI (no necesita key).

## 1. Setup




In [1]:
%pip install -q "langchain>=1.0" "langchain-openai" "langgraph>=1.0" requests pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 1.2. Cargar la API key de OpenAI



In [2]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")


## 2. Las tools

Las dos tools estan en `tools.py`. Primero las probamos sueltas con `.invoke({...})` para ver que la API responde.

In [3]:
from tools import get_pokemon_info, get_pokemon_moves, HERRAMIENTAS

print(get_pokemon_info.invoke({"name": "tyranitar"}))
print(get_pokemon_moves.invoke({"name": "charizard"}))


Tyranitar | tipos: rock, dark | stats: {'hp': 100, 'attack': 134, 'defense': 110, 'special-attack': 95, 'special-defense': 100, 'speed': 61}
Charizard aprende 131 movimientos. Algunos: mega-punch, fire-punch, thunder-punch, scratch, swords-dance, cut, wing-attack, fly, mega-kick, headbutt, body-slam, take-down


## 3. El agente

Lo montamos con `create_agent` de `langchain.agents`. Le pasamos el modelo, las tools y un `system_prompt` que le dice cuando usar cada una.

In [4]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

#Creamos el agente con el modelo y las herramientas
PROMPT_SISTEMA = (
    "Eres un asesor de incursiones (raids) de Pokemon GO. Tienes dos herramientas: "
    "get_pokemon_info y get_pokemon_moves.\n\n"
    "Como actuar segun la pregunta:\n"
    "1. Si te preguntan como derrotar a un Pokemon en una raid (counters, a que es debil, "
    "que usar contra X): llama a get_pokemon_info del jefe para obtener sus tipos y, a "
    "partir de la tabla de efectividad de tipos que ya conoces, deduce sus debilidades y "
    "recomienda 3-5 Pokemon counter concretos (Tyranitar, Rayquaza, Garchomp, Metagross, "
    "Dragonite, Gengar, Lucario, Mamoswine, etc.) de esos tipos. Puedes usar "
    "get_pokemon_moves sobre los counters para sugerir un buen moveset.\n"
    "2. Si te preguntan tipos o stats de un Pokemon: usa get_pokemon_info.\n"
    "3. Si te preguntan movimientos de un Pokemon concreto: usa get_pokemon_moves.\n"
    "4. Para preguntas generales del juego (que es una raid, como funcionan los pases, etc.) "
    "responde directamente sin herramientas.\n\n"
    "Responde en espanol, se conciso y no te inventes datos de tipos o movimientos."
)

modelo = ChatOpenAI(model="gpt-5-mini")
agente = create_agent(model=modelo, tools=HERRAMIENTAS, system_prompt=PROMPT_SISTEMA)


c:\Users\Usuario\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Pregunta concreta (deberia llamar a una tool)

Con `agente.stream(...)` se ve el bucle ReAct paso a paso: el agente decide la accion (la tool), recibe los datos y luego da la respuesta.

In [5]:
def ejecutar(pregunta):
    print(f"PREGUNTA: {pregunta}\n")
    for fragmento in agente.stream({"messages": [{"role": "user", "content": pregunta}]}, stream_mode="values"):
        fragmento["messages"][-1].pretty_print()


ejecutar("Tengo una incursion de Rayquaza. Que deberia usar para derrotarlo?")


PREGUNTA: Tengo una incursion de Rayquaza. Que deberia usar para derrotarlo?

================================ Human Message =================================

Tengo una incursion de Rayquaza. Que deberia usar para derrotarlo?
================================== Ai Message ==================================
Tool Calls:
  get_pokemon_info (call_VTTuQ5jnYv7tOy4iUhYlm7dt)
 Call ID: call_VTTuQ5jnYv7tOy4iUhYlm7dt
  Args:
    name: Rayquaza
  get_pokemon_moves (call_3GkDOsVmQ7QXBaBJjVJAjCbc)
 Call ID: call_3GkDOsVmQ7QXBaBJjVJAjCbc
  Args:
    name: Mamoswine
  get_pokemon_moves (call_5tZ8OEGxbp1MI8t0SFSXKdjs)
 Call ID: call_5tZ8OEGxbp1MI8t0SFSXKdjs
  Args:
    name: Tyranitar
  get_pokemon_moves (call_5baFBM2J868kw3iE1uQB0oHo)
 Call ID: call_5baFBM2J868kw3iE1uQB0oHo
  Args:
    name: Garchomp
  get_pokemon_moves (call_N7CZALQmXG4xOhJmvVFwpPaZ)
 Call ID: call_N7CZALQmXG4xOhJmvVFwpPaZ
  Args:
    name: Dragonite
  get_pokemon_moves (call_S7qTdXYWPi2UWlfdX5vAVomU)
 Call ID: call_S7qTdXYWPi2UWlfd

### Pregunta general (no deberia llamar a tools)

Esto no necesita datos de la PokeAPI, asi que el agente deberia contestar directamente.

In [6]:
ejecutar("Que es una incursion (raid) en Pokemon GO?")


PREGUNTA: Que es una incursion (raid) en Pokemon GO?

================================ Human Message =================================

Que es una incursion (raid) en Pokemon GO?
================================== Ai Message ==================================

Una incursión (raid) en Pokémon GO es un encuentro cooperativo contra un Pokémon jefe en un Gimnasio. Resumen rápido y práctico:

- Qué es: un jefe de raid aparece en un Gimnasio (a veces dentro de un huevo antes de salir). Los jugadores se unen para derrotarlo en una batalla en tiempo real y después intentar capturarlo.
- Cómo unirse: necesitas un Raid Pass (gratuito 1 al día o Pase Remoto para unirte a distancia). Puedes unirte desde el mapa cuando hay un huevo o el icono del jefe en el Gimnasio, o recibir una invitación de amigos.
- Tipos de raids: hay varios niveles (Tier 1–5), raids legendarios (Tier 5) y raids Mega. Los de menor nivel son fáciles en solitario; los legendarios y Mega suelen requerir varios jugadores.
- Mecán

## 4. Evaluacion

Los casos de prueba estan en `evaluation.py`: nueve preguntas, siete que requieren tool y dos generales que no deberian disparar ninguna. Para cada caso registramos la tool esperada, la tool usada, si el agente acerto, la duracion y la respuesta. La funcion `accuracy` calcula el porcentaje de aciertos.
Estos casos constituyen un pequeño Golden Dataset utilizado para validar el comportamiento del agente y la correcta selección de herramientas.

In [8]:
import pandas as pd
from evaluation import evaluar, accuracy

filas = evaluar(agente)
metricas = accuracy(filas)
print(f"Accuracy: {metricas['accuracy']:.0%} ({metricas['aciertos']}/{metricas['total']} casos correctos)")
pd.DataFrame(filas)

Accuracy: 100% (9/9 casos correctos)


,pregunta,tool_esperada,tools_usadas,correcto,duracion_s,respuesta
0,A que es debil Dragonite?,get_pokemon_info,get_pokemon_info,True,9.96,Dragonite es tipo Dragón/Volador. Sus debilida...
1,Que tipo es Tyranitar?,get_pokemon_info,get_pokemon_info,True,4.75,Tyranitar es de tipo Roca y Siniestro.
2,Que movimientos aprende Charizard?,get_pokemon_moves,get_pokemon_moves,True,6.91,Charizard aprende 131 movimientos. Algunos eje...
3,"Tengo una raid de Rayquaza, que uso?",get_pokemon_info,"get_pokemon_info, get_pokemon_moves, get_pokem...",True,52.77,Perfecto — Rayquaza es Dragón/Volador. Debilid...
4,Cuales son los stats base de Metagross?,get_pokemon_info,get_pokemon_info,True,4.18,Metagross — tipos: Acero / Psíquico.\n\nStats ...
5,Dime algun ataque que aprenda Gengar,get_pokemon_moves,get_pokemon_moves,True,7.29,Algunos ataques que aprende Gengar son: fire-p...
6,Que counters uso contra una raid de Groudon?,get_pokemon_info,"get_pokemon_info, get_pokemon_moves, get_pokem...",True,58.41,Resumen rápido\n- Groudon: tipo Ground.\n- Déb...
7,Que es una incursion en Pokemon GO?,(ninguna),(ninguna),True,10.61,Una incursión (raid) en Pokémon GO es un comba...
8,Cuantos jugadores hacen falta para una raid de...,(ninguna),(ninguna),True,6.79,Depende del jefe y de cuán preparados estén lo...


## Conclusiones

El agente funciona como se esperaba: llama a la tool adecuada para preguntas concretas de un Pokemon y responde directamente cuando la pregunta es general del juego. Sobre el Golden Dataset de nueve casos obtiene un 100% de accuracy en la seleccion de herramientas. Las preguntas simples (un solo dato) se resuelven en unos 4-8 segundos, mientras que las de counters de raid, donde el agente encadena varias llamadas a las tools para razonar debilidades y movesets, tardan mas (40-60 segundos) con gpt-5-mini.